# # Linear Regression Models for NYC Taxi Trip Duration Prediction

---

## Project

**NYC Taxi Trip Duration Prediction**

This notebook trains and evaluates the required baseline and linear regression models for predicting NYC Yellow Taxi trip duration.

The dataset used in this notebook is the **feature-engineered dataset** produced during the previous phase of the project. The objective is to establish strong baseline models before moving to more advanced tree-based and ensemble methods.

---

# Objectives

This notebook will:

- Load the engineered dataset
- Perform a time-based train/validation/test split
- Prepare features and target variable
- Scale numerical features for linear models
- Train the required baseline model
- Train multiple linear regression models
- Evaluate every model using identical metrics
- Compare model performance
- Save trained models
- Save evaluation results

---

# Models

Required models:

1. Dummy Median Baseline
2. Multiple Linear Regression
3. Ridge Regression
4. Lasso Regression
5. Elastic Net Regression

---

# Evaluation Metrics

Each model will be evaluated using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Median Absolute Error
- Coefficient of Determination (R²)
- Training Time
- Prediction Time

---

# Output

Generated artifacts:

```
models/
├── dummy_baseline.joblib
├── linear_regression.joblib
├── ridge_regression.joblib
├── lasso_regression.joblib
├── elastic_net.joblib
└── best_linear_model.joblib

reports/
└── linear_results.csv
```

---

# Workflow

```
Feature Dataset
        │
        ▼
Load Dataset
        │
        ▼
Train / Validation / Test Split
        │
        ▼
Feature Scaling
        │
        ▼
Train Models
        │
        ▼
Evaluate Models
        │
        ▼
Compare Results
        │
        ▼
Save Models & Reports
```

In [1]:


from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd

# Machine Learning

from sklearn.preprocessing import StandardScaler

# =============================================================================
# Project Paths
# =============================================================================

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"

MODELS_DIR.mkdir(exist_ok=True)
REPORTS_DIR.mkdir(exist_ok=True)

print(f"Project Root : {PROJECT_ROOT}")

Project Root : c:\Users\abuba\Downloads\ML-Projects\Day-5


## Import Project Modules

In [2]:
from src.data.load_data import load_zone_lookup

from src.features.build_features import load_featured_data

from src.models.train import (
    train_dummy_baseline,
    train_linear_regression,
    train_ridge_regression,
    train_lasso_regression,
    train_elastic_net_regression,
)

from src.models.evaluate import (
    evaluate_model,
    create_results_dataframe,
)

from src.models.utils import (
    save_model,
    save_results,
)

## Load Feature-Engineered Dataset

In [3]:
# =============================================================================
# Load Feature-Engineered Dataset
# =============================================================================

featured_df = load_featured_data()

print("=" * 70)
print("Feature Dataset Loaded Successfully")
print("=" * 70)

print(f"Rows    : {featured_df.shape[0]:,}")
print(f"Columns : {featured_df.shape[1]}")

featured_df.head()

Feature Dataset Loaded Successfully
Rows    : 18,378,485
Columns : 32


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,congestion_surcharge,...,hour_sin,hour_cos,day_sin,day_cos,same_borough,same_location,is_airport_trip,distance_squared,log_trip_distance,distance_per_passenger
0,2,2023-01-01 00:32:10,2023-01-01 00:40:36,1,0.97,1.0,N,161,141,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,0.9409,0.678034,0.97
1,2,2023-01-01 00:55:08,2023-01-01 01:01:27,1,1.10,1.0,N,43,237,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,1.2100,0.741937,1.10
2,2,2023-01-01 00:25:04,2023-01-01 00:37:49,1,2.51,1.0,N,48,238,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,6.3001,1.255616,2.51
3,2,2023-01-01 00:10:29,2023-01-01 00:21:19,1,1.43,1.0,N,107,79,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,2.0449,0.887891,1.43
4,2,2023-01-01 00:50:34,2023-01-01 01:02:52,1,1.84,1.0,N,161,137,2.5,...,0.0,1.0,-0.781832,0.62349,1,0,0,3.3856,1.043804,1.84


In [4]:
# =============================================================================
# Dataset Information
# =============================================================================

featured_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18378485 entries, 0 to 18378484
Data columns (total 32 columns):
 #   Column                  Dtype         
---  ------                  -----         
 0   VendorID                int64         
 1   tpep_pickup_datetime    datetime64[us]
 2   tpep_dropoff_datetime   datetime64[us]
 3   passenger_count         int8          
 4   trip_distance           float32       
 5   RatecodeID              float64       
 6   store_and_fwd_flag      str           
 7   PULocationID            int64         
 8   DOLocationID            int64         
 9   congestion_surcharge    float64       
 10  airport_fee             float64       
 11  trip_duration_minutes   float32       
 12  log_trip_duration       float64       
 13  pickup_year             int16         
 14  pickup_month            int8          
 15  pickup_day              int8          
 16  pickup_hour             int8          
 17  pickup_dayofweek        int8          
 18  pickup_week

In [5]:
# =============================================================================
# Target Variable
# =============================================================================

TARGET = "log_trip_duration"

print(f"Target Column : {TARGET}")

featured_df[TARGET].describe()

Target Column : log_trip_duration


count    1.837848e+07
mean     2.594465e+00
std      6.705067e-01
min      1.652930e-02
25%      2.142025e+00
50%      2.582739e+00
75%      3.032546e+00
max      5.887446e+00
Name: log_trip_duration, dtype: float64

In [6]:
# =============================================================================
# Prepare Features and Target
# =============================================================================

TARGET = "log_trip_duration"

# Columns not used for training
DROP_COLUMNS = [
    "log_trip_duration",        # Target
    "trip_duration_minutes",    # Original target
    "tpep_pickup_datetime",     # Raw datetime
    "tpep_dropoff_datetime",    # Causes data leakage
]

X = featured_df.drop(columns=DROP_COLUMNS)

y = featured_df[TARGET]

print("=" * 70)
print("Dataset Prepared for Training")
print("=" * 70)
print(f"Feature Matrix Shape : {X.shape}")
print(f"Target Shape         : {y.shape}")
print(f"Number of Features   : {X.shape[1]}")

X.head()

Dataset Prepared for Training
Feature Matrix Shape : (18378485, 28)
Target Shape         : (18378485,)
Number of Features   : 28


,VendorID,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,congestion_surcharge,airport_fee,pickup_year,...,hour_sin,hour_cos,day_sin,day_cos,same_borough,same_location,is_airport_trip,distance_squared,log_trip_distance,distance_per_passenger
0,2,1,0.97,1.0,N,161,141,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,0.9409,0.678034,0.97
1,2,1,1.10,1.0,N,43,237,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,1.2100,0.741937,1.10
2,2,1,2.51,1.0,N,48,238,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,6.3001,1.255616,2.51
3,2,1,1.43,1.0,N,107,79,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,2.0449,0.887891,1.43
4,2,1,1.84,1.0,N,161,137,2.5,0.0,2023,...,0.0,1.0,-0.781832,0.62349,1,0,0,3.3856,1.043804,1.84


## Train / Validation / Test Split

In [7]:
# =============================================================================
# Train / Validation / Test Split
# =============================================================================

TRAIN_MONTHS = [1, 2, 3, 4, 5]
VALID_MONTH = 6
TEST_MONTH = 7

train_mask = featured_df["pickup_month"].isin(TRAIN_MONTHS)
valid_mask = featured_df["pickup_month"] == VALID_MONTH
test_mask = featured_df["pickup_month"] == TEST_MONTH

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_valid = X.loc[valid_mask].copy()
y_valid = y.loc[valid_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

## Verify Split

In [8]:
print("Training Months  :", sorted(X_train["pickup_month"].unique()))
print("Validation Month :", sorted(X_valid["pickup_month"].unique()))
print("Test Month       :", sorted(X_test["pickup_month"].unique()))

Training Months  : [np.int8(1), np.int8(2), np.int8(3), np.int8(4), np.int8(5)]
Validation Month : [np.int8(6)]
Test Month       : [np.int8(7)]


## Check Feature Data Types

In [9]:
print(X_train.dtypes)

VendorID                    int64
passenger_count              int8
trip_distance             float32
RatecodeID                float64
store_and_fwd_flag            str
PULocationID                int64
DOLocationID                int64
congestion_surcharge      float64
airport_fee               float64
pickup_year                 int16
pickup_month                 int8
pickup_day                   int8
pickup_hour                  int8
pickup_dayofweek             int8
pickup_week                 int16
pickup_weekend               int8
pickup_quarter               int8
rush_hour                    int8
hour_sin                  float32
hour_cos                  float32
day_sin                   float32
day_cos                   float32
same_borough                 int8
same_location                int8
is_airport_trip              int8
distance_squared          float32
log_trip_distance         float32
distance_per_passenger    float32
dtype: object


## Convert store_and_fwd_flag

In [10]:
mapping = {
    "N": 0,
    "Y": 1
}

X_train["store_and_fwd_flag"] = (
    X_train["store_and_fwd_flag"]
    .map(mapping)
    .astype("int8")
)

X_valid["store_and_fwd_flag"] = (
    X_valid["store_and_fwd_flag"]
    .map(mapping)
    .astype("int8")
)

X_test["store_and_fwd_flag"] = (
    X_test["store_and_fwd_flag"]
    .map(mapping)
    .astype("int8")
)

## Scale Features

In [11]:


from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_valid_scaled = scaler.transform(X_valid)

X_test_scaled = scaler.transform(X_test)

print("=" * 70)
print("Feature Scaling Completed")
print("=" * 70)

print(f"Training : {X_train_scaled.shape}")
print(f"Validation : {X_valid_scaled.shape}")
print(f"Test : {X_test_scaled.shape}")

Feature Scaling Completed
Training : (15270098, 28)
Validation : (3108362, 28)
Test : (22, 28)


## Dummy Median Baseline

In [13]:
dummy_model, dummy_train_time = train_dummy_baseline(
    X_train_scaled,
    y_train,
)

print("=" * 70)
print("Dummy Median Baseline Trained Successfully")
print("=" * 70)

print(f"Training Time : {dummy_train_time:.4f} seconds")

2026-07-23 19:25:02,083 | INFO | Training Dummy Median Baseline...
2026-07-23 19:25:02,288 | INFO | Completed in 0.204 seconds


Dummy Median Baseline Trained Successfully
Training Time : 0.2041 seconds


## Evaluate Model

In [14]:
dummy_results = evaluate_model(
    model=dummy_model,
    X_test=X_test_scaled,
    y_test=y_test,
    training_time=dummy_train_time,
    model_name="Dummy Median Baseline",
)

dummy_results

2026-07-23 19:25:07,774 | INFO | Evaluating Dummy Median Baseline
2026-07-23 19:25:08,175 | INFO | Dummy Median Baseline evaluation completed


{'Model': 'Dummy Median Baseline',
 'MAE': 0.5387868269900324,
 'RMSE': 0.6529625726263529,
 'Median AE': 0.5112392741483982,
 'R²': -0.23541561912527098,
 'Training Time (s)': 0.20406469999579713,
 'Prediction Time (s)': 0.027460499986773357}

## Store Results

In [15]:
results = []

results.append(dummy_results)

pd.DataFrame(results)

,Model,MAE,RMSE,Median AE,R²,Training Time (s),Prediction Time (s)
0,Dummy Median Baseline,0.538787,0.652963,0.511239,-0.235416,0.204065,0.02746


## Multiple Linear Regression

In [19]:
linear_model, linear_train_time = train_linear_regression(
    X_train_scaled,
    y_train,
)

print("=" * 70)
print("Multiple Linear Regression Trained Successfully")
print("=" * 70)

print(f"Training Time : {linear_train_time:.4f} seconds")

2026-07-22 21:55:05,188 | INFO | Training Linear Regression...
2026-07-22 21:55:28,175 | INFO | Completed in 22.982 seconds


Multiple Linear Regression Trained Successfully
Training Time : 22.9819 seconds


## Evaluate Model

In [20]:
linear_results = evaluate_model(
    model=linear_model,
    X_test=X_test_scaled,
    y_test=y_test,
    training_time=linear_train_time,
    model_name="Multiple Linear Regression",
)

linear_results

2026-07-22 21:55:29,887 | INFO | Evaluating Multiple Linear Regression...
2026-07-22 21:55:29,922 | INFO | Multiple Linear Regression evaluation completed.


{'Model': 'Multiple Linear Regression',
 'MAE': 0.22253804455895293,
 'RMSE': 0.2716211143779465,
 'Median AE': 0.23793699165299786,
 'R²': 0.7862217271492031,
 'Training Time (s)': 22.981888499998604,
 'Prediction Time (s)': 0.005692099992302246}

## Save Model

In [21]:
save_model(
    model=linear_model,
    model_name="linear_regression",
    models_dir=MODELS_DIR,
)

2026-07-22 21:55:49,050 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\linear_regression.joblib


WindowsPath('c:/Users/abuba/Downloads/ML-Projects/Day-5/models/linear_regression.joblib')

In [22]:
# =============================================================================
# Update Results
# =============================================================================

results.append(linear_results)

pd.DataFrame(results)

,Model,MAE,RMSE,Median AE,R²,Training Time (s),Prediction Time (s)
0,Dummy Median Baseline,0.538787,0.652963,0.511239,-0.235416,0.954260,0.003930
1,Multiple Linear Regression,0.222538,0.271621,0.237937,0.786222,22.981888,0.005692


## Ridge Regression

In [23]:
ridge_model, ridge_train_time = train_ridge_regression(
    X_train_scaled,
    y_train,
)

print("=" * 70)
print("Ridge Regression Trained Successfully")
print("=" * 70)

print(f"Training Time : {ridge_train_time:.4f} seconds")

2026-07-22 21:56:34,637 | INFO | Training Ridge Regression (alpha=1.0)...
2026-07-22 21:56:40,562 | INFO | Completed in 5.923 seconds


Ridge Regression Trained Successfully
Training Time : 5.9231 seconds


## Evaluate Ridge Regression

In [24]:
ridge_results = evaluate_model(
    model=ridge_model,
    X_test=X_test_scaled,
    y_test=y_test,
    training_time=ridge_train_time,
    model_name="Ridge Regression",
)

ridge_results

2026-07-22 21:56:54,004 | INFO | Evaluating Ridge Regression...
2026-07-22 21:56:54,008 | INFO | Ridge Regression evaluation completed.


{'Model': 'Ridge Regression',
 'MAE': 0.22253800091466538,
 'RMSE': 0.2716210701064112,
 'Median AE': 0.23793698921508755,
 'R²': 0.7862217968366544,
 'Training Time (s)': 5.9231156000023475,
 'Prediction Time (s)': 0.000598000013269484}

In [25]:
# =============================================================================
# Save Ridge Regression Model
# =============================================================================

save_model(
    model=ridge_model,
    model_name="ridge_regression",
    models_dir=MODELS_DIR,
)

2026-07-22 21:57:07,854 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\ridge_regression.joblib


WindowsPath('c:/Users/abuba/Downloads/ML-Projects/Day-5/models/ridge_regression.joblib')

In [26]:
# =============================================================================
# Update Model Comparison
# =============================================================================

results.append(ridge_results)

comparison_df = pd.DataFrame(results)

comparison_df

,Model,MAE,RMSE,Median AE,R²,Training Time (s),Prediction Time (s)
0,Dummy Median Baseline,0.538787,0.652963,0.511239,-0.235416,0.954260,0.003930
1,Multiple Linear Regression,0.222538,0.271621,0.237937,0.786222,22.981888,0.005692
2,Ridge Regression,0.222538,0.271621,0.237937,0.786222,5.923116,0.000598


## Lasso Regression

In [27]:
lasso_model, lasso_train_time = train_lasso_regression(
    X_train_scaled,
    y_train,
)

print("=" * 70)
print("Lasso Regression Trained Successfully")
print("=" * 70)

print(f"Training Time : {lasso_train_time:.4f} seconds")

2026-07-22 21:58:12,440 | INFO | Training Lasso Regression (alpha=0.001)...
2026-07-22 21:59:23,085 | INFO | Completed in 70.644 seconds


Lasso Regression Trained Successfully
Training Time : 70.6445 seconds


## Evaluate Lasso Regression

In [28]:
lasso_results = evaluate_model(
    model=lasso_model,
    X_test=X_test_scaled,
    y_test=y_test,
    training_time=lasso_train_time,
    model_name="Lasso Regression",
)

lasso_results

2026-07-22 21:59:23,233 | INFO | Evaluating Lasso Regression...
2026-07-22 21:59:23,241 | INFO | Lasso Regression evaluation completed.


{'Model': 'Lasso Regression',
 'MAE': 0.2222105405854937,
 'RMSE': 0.27122128825546726,
 'Median AE': 0.23329601978608094,
 'R²': 0.7868506269249749,
 'Training Time (s)': 70.64448830000765,
 'Prediction Time (s)': 0.00039960000140126795}

## Save Lasso Model

In [29]:
save_model(
    model=lasso_model,
    model_name="lasso_regression",
    models_dir=MODELS_DIR,
)

2026-07-22 21:59:23,261 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\lasso_regression.joblib


WindowsPath('c:/Users/abuba/Downloads/ML-Projects/Day-5/models/lasso_regression.joblib')

In [30]:
results.append(lasso_results)

comparison_df = pd.DataFrame(results)

comparison_df

,Model,MAE,RMSE,Median AE,R²,Training Time (s),Prediction Time (s)
0,Dummy Median Baseline,0.538787,0.652963,0.511239,-0.235416,0.954260,0.003930
1,Multiple Linear Regression,0.222538,0.271621,0.237937,0.786222,22.981888,0.005692
2,Ridge Regression,0.222538,0.271621,0.237937,0.786222,5.923116,0.000598
3,Lasso Regression,0.222211,0.271221,0.233296,0.786851,70.644488,0.000400


## Elastic Net Regression

Elastic Net combines L1 (Lasso) and L2 (Ridge) regularization, making it a balanced approach when features are correlated while also performing feature selection.

In [31]:
elastic_model, elastic_train_time = train_elastic_net_regression(
    X_train_scaled,
    y_train,
)

print("=" * 70)
print("Elastic Net Regression Trained Successfully")
print("=" * 70)

print(f"Training Time : {elastic_train_time:.4f} seconds")

2026-07-22 22:00:43,676 | INFO | Training Elastic Net (alpha=0.001, l1_ratio=0.5)...
2026-07-22 22:02:10,063 | INFO | Completed in 86.386 seconds


Elastic Net Regression Trained Successfully
Training Time : 86.3855 seconds


## Evaluate Elastic Net Regression

In [32]:
elastic_results = evaluate_model(
    model=elastic_model,
    X_test=X_test_scaled,
    y_test=y_test,
    training_time=elastic_train_time,
    model_name="Elastic Net Regression",
)

elastic_results

2026-07-22 22:02:10,121 | INFO | Evaluating Elastic Net Regression...
2026-07-22 22:02:10,125 | INFO | Elastic Net Regression evaluation completed.


{'Model': 'Elastic Net Regression',
 'MAE': 0.2220232466666217,
 'RMSE': 0.2709863915494091,
 'Median AE': 0.23475066726860083,
 'R²': 0.7872196717349474,
 'Training Time (s)': 86.38553440000396,
 'Prediction Time (s)': 0.000509100005729124}

## Save Elastic Net Model

In [33]:
save_model(
    model=elastic_model,
    model_name="elastic_net_regression",
    models_dir=MODELS_DIR,
)

2026-07-22 22:02:10,748 | INFO | Model saved to: c:\Users\abuba\Downloads\ML-Projects\Day-5\models\elastic_net_regression.joblib


WindowsPath('c:/Users/abuba/Downloads/ML-Projects/Day-5/models/elastic_net_regression.joblib')

In [ ]:
results.append(elastic_results)

comparison_df = pd.DataFrame(results)

comparison_df

# Conclusion

In this notebook, we successfully established the baseline and linear regression models for the **NYC Taxi Trip Duration Prediction** project. The feature-engineered dataset was loaded, prepared using a time-based train/validation/test split, and standardized for algorithms that require feature scaling.

The following required regression models were trained and evaluated using identical evaluation metrics to ensure a fair comparison:

- ✅ Dummy Median Baseline
- ✅ Multiple Linear Regression
- ✅ Ridge Regression
- ✅ Lasso Regression
- ✅ Elastic Net Regression

Each model was assessed using:

- Mean Absolute Error (MAE)
- Root Mean Squared Error (RMSE)
- Median Absolute Error
- Coefficient of Determination (R²)
- Training Time
- Prediction Time

The trained models and their evaluation results were saved for future comparison and reproducibility. These linear models provide a strong performance baseline and establish a benchmark against which more advanced machine learning algorithms will be evaluated.

The next notebook, **05_tree_based_models.ipynb**, will focus on training and evaluating tree-based regression models, including:

- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor (or HistGradientBoostingRegressor)

These models are expected to better capture the non-linear relationships present in taxi trip duration data and may achieve improved predictive performance over the linear models.